In [ ]:
import ray
from ray.job_submission import JobSubmissionClient
import time
from ray.runtime_env import RuntimeEnv

In [ ]:
# Ray cluster information
ray_head_ip = "kuberay-head-svc.kuberay.svc.cluster.local"
ray_head_port = 8265
ray_address = f"http://{ray_head_ip}:{ray_head_port}"

In [ ]:
# Submit Ray job using JobSubmissionClient
client = JobSubmissionClient(ray_address)
job_id = client.submit_job(
    entrypoint="python mandlebrot.py",
    runtime_env={
        "working_dir": "./",
        "pip": ["piexif","ipyplot","whylogs[image,whylabs]", "matplotlib==3.9.0"],
        "env_vars": {"http_proxy":"<ADD-http-proxy>","https_proxy":"<ADD-https-proxy>"}
    },
    entrypoint_num_cpus=3
)

print(f"Ray job submitted with job_id: {job_id}")

# Waiting for Ray to finish the job and print the result
while True:
    status = client.get_job_status(job_id)
    if status in [ray.job_submission.JobStatus.RUNNING, ray.job_submission.JobStatus.PENDING]:
        time.sleep(5)
    else:
        break
try:
    logs = client.get_job_logs(job_id) 
    print(logs)
except RuntimeError as e:
    print(f"Failed to get job logs, please check logs on ray dashboard ")